<a href="https://colab.research.google.com/github/lsteffenel/M2Atmo_et_Climat/blob/main/Coccinelle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prédiction d'une trajectoire 2D avec un réseau récurrent (GRU)

## Objectifs
- Comprendre comment un réseau récurrent apprend la **dynamique** d'une série temporelle.
- Construire soi-même le pipeline : données → fenêtres → modèle → entraînement → prédiction.
- Découvrir la différence entre une prédiction à 1 pas et une prédiction à n pas (et pourquoi la seconde dérive).

## Fil rouge
Une coccinelle se déplace selon une trajectoire pseudo-aléatoire (somme de sinusoïdes).

On observe 20 positions consécutives : saurez-vous prédire les 5 suivantes ?

C'est exactement le schéma d'une prévision météorologique : prédire l'état futurdu système à partir de son histoire récente.

## Comment travailler
Les cellules marquées **À VOUS** contiennent des `TODO` : complétez-les avant d'exécuter.

Les cellules **Question** se répondent dans une cellule de texte, avant de lancer la suite.

Ne lisez pas la cellule "Éléments de réponse" avant d'avoir essayé.

## Étape 1 — Imports
On n'importe que le strict nécessaire. Pas de librairie de plot "magique" : tout se fait avec matplotlib.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.sequence import TimeseriesGenerator

print(tf.__version__)

## Étape 2 — Générer la trajectoire

La position (x, y) de la coccinelle à l'instant t est la somme de 8 sinusoïdes (4 par axe), d'amplitudes et de fréquences tirées au hasard :

$$x(t) = \sum_i a_i \sin(t\,\omega_i) \quad \text{(idem pour } y \text{)}$$

Le résultat est régulier mais imprévisible sans modèle : parfait pour notre exercice.

In [ ]:
def ladybug_init(s=122):
    """Tire au hasard les 8 paramètres (amplitudes + fréquences) de chaque axe."""
    if s > 0:
        random.seed(s)
    ladybug_init.params_x = [random.gauss(0., 1.) for _ in range(8)]
    ladybug_init.params_y = [random.gauss(0., 1.) for _ in range(8)]

def ladybug_move(t):
    """Position (x, y) de la coccinelle à l'instant t."""
    [ax1, ax2, ax3, ax4, kx1, kx2, kx3, kx4] = ladybug_init.params_x
    [ay1, ay2, ay3, ay4, ky1, ky2, ky3, ky4] = ladybug_init.params_y
    x = ax1*np.sin(t*(kx1+20)) + ax2*np.cos(t*(kx2+10)) + ax3*np.sin(t*(kx3+5)) + ax4*np.cos(t*(kx4+5))
    y = ay1*np.cos(t*(ky1+20)) + ay2*np.sin(t*(ky2+10)) + ay3*np.cos(t*(ky3+5)) + ay4*np.sin(t*(ky4+5))
    return x, y

## Étape 3 — Construire le jeu de données

La cellule suivante fait tout : échantillonnage temporel, découpage
entraînement/test, normalisation. Exécutez, puis répondez aux questions.

In [ ]:
max_t, delta_t   = 1000, 0.02      # durée simulée et pas de temps
sequence_len     = 20             # longueur de la fenêtre d'entrée
predict_len      = 5              # horizon de prédiction
train_prop      = 0.8
batch_size       = 32
EPOCHS           = 10

ladybug_init(s=16)

positions = []
x, y = 0, 0
for t in np.arange(0., max_t, delta_t):
    positions.append([x, y])
    x, y = ladybug_move(t)

dataset = np.array(positions)
k = int(len(dataset) * train_prop)
x_train, x_test = dataset[:k], dataset[k:]

# Normalisation : statistiques du jeu d'entraînement uniquement,
# appliquées aux deux jeux.
mean, std = x_train.mean(), x_train.std()
x_train = (x_train - mean) / std
x_test  = (x_test - mean) / std

print('train :', x_train.shape, '   test :', x_test.shape)

plt.figure(figsize=(7, 7))
plt.plot(x_train[:100, 0], x_train[:100, 1], lw=0.5) # Affiche les 100 premiers "pas" de la coccinnelle
plt.axis('equal')
plt.title("Trajectoire d'entraînement (normalisée)")
plt.show()

### Question 1
1. Pourquoi normalise-t-on avec les statistiques de `x_train` uniquement,
alors que la normalisation est appliquée aux deux jeux ? Quel serait le biais
si on utilisait tout le dataset ?
2. Ici la normalisation est faite composante par composante (`x_train.mean()`
sur tout le tableau). Seriez-vous d'accord avec cette approche pour des données atmosphériques (température, pression, humidité) ? Que ferait un z-score par variable, et pourquoi est-ce indispensable dès que les unités diffèrent ?
3. Combien y a-t-il de points temporels dans `x_train` ? Et combien de
fenêtres de 20 points peut-on y découper ?

## Étape 4 — Fenêtres temporelles
Un réseau récurrent n'avale pas toute la série : il lit une fenêtre de
`sequence_len` points et doit prédire le point suivant. `TimeseriesGenerator`
fabrique automatiquement ces couples (fenêtre → cible), par lots de `batch_size`.

In [ ]:
train_generator = TimeseriesGenerator(x_train, x_train, length=sequence_len, batch_size=batch_size)
test_generator  = TimeseriesGenerator(x_test,  x_test,  length=sequence_len,  batch_size=batch_size)

x, y = train_generator[0]
print('Nombre de lots :', len(train_generator))
print('x ->', x.shape, '   y ->', y.shape)
print('Début d une fenêtre :\n', np.round(x[0][:3], 2), ' ...')
print('Sa cible          :', np.round(y[0], 2))

## Question 2
1. Vérifiez la cohérence : `y[0]` devrait être le point qui suit immédiatement la fenêtre `x[0]`. Confirmez-le avec un print. Quelle est la différence avec la cible `y[0][1]` (2e point du lot) ?
2. Expliquez pourquoi il y a `len(x_train) - sequence_len` fenêtres possibles
et non `len(x_train)`. Consequence pour la taille du dataset ?
3. Les fenêtres se chevauchent presque entièrement. Pourquoi n'est-ce pas
gênant pour l'entraînement, mais pourquoi faut-il malgré tout un jeu de
 test disjoint (et temporellement postérieur) ?

##Étape 5 — Le modèle GRU

Une cellule GRU (Gated Recurrent Unit) maintient un état caché mis à jour à
chaque pas de temps : elle "lit" la fenêtre point par point et condense la
dynamique dans son état final. Deux portes (mise à jour, réinitialisation)
contrôlent ce qu'il faut mémoriser ou oublier — une forme de mémoire sélective.

Le modèle ci-dessous fonctionne tel quel.

In [ ]:
model = keras.models.Sequential([
    keras.layers.Input(shape=(sequence_len, 2)),
    keras.layers.GRU(200, return_sequences=False),
    keras.layers.Dense(2),
])
model.summary()

model.compile(optimizer='rmsprop', loss='mse', metrics=['mae'])

### EXPLORATION 1 — Architecture
Avant d'entraîner, notez vos prédictions par écrit, puis testez.
1. `model.summary()` donne ~243 000 paramètres pour la couche GRU.
   La formule d'une GRU est `3 * units * (units + features + 1)`
   (3 blocs internes, cf. documentation Keras — à vérifier). Retrouvez ce nombre.
2. Réduisez à `GRU(16)` : le nombre de paramètres chute-t-il comme vous l'attendiez ?
   Entraînez : la perte finale change-t-elle vraiment ?
3. Remplacez la GRU par un `SimpleRNN(200)` puis un `LSTM(200)` : comparez
   nombre de paramètres, vitesse d'entraînement et qualité de prédiction.
   Qu'est-ce qui, dans la physique du problème (somme de sinusoïdes lentes),
   explique que l'écart soit petit ici ?
4. Essayez `return_sequences=True` avec une 2e couche GRU par-dessus :
   quand cette option est-elle nécessaire ?

### EXPLORATION 2 — Activation et optimiseur
5. Ajoutez `activation='relu'` à la GRU. Le GRU utilise `tanh` par défaut :
    quel effet observez-vous sur la stabilité de l'entraînement ?
6. Testez `optimizer='adam'` au lieu de `'rmsprop'`, avec le même nombre
    d'époques. Lequel converge le plus vite ? Lequel généralise mieux sur le test ?

## Étape 6 — Entraînement

In [ ]:
history = model.fit(train_generator,
                    epochs=EPOCHS,
                    validation_data=test_generator,
                    verbose=1)

# %%
plt.figure(figsize=(8, 5))
plt.plot(history.history['loss'], label='entraînement')
plt.plot(history.history['val_loss'], label='validation')
plt.xlabel('Époque')
plt.ylabel('MSE')
plt.legend()
plt.show()


### EXPLORATION 3 — Entraînement
1. La courbe `val_loss` suit-elle `loss` ? À partir de combien d'époques
   le gain devient marginal ? Testez `EPOCHS = 30`.
2. Testez `batch_size = 8` et `batch_size = 256`. Effet sur le nombre de lots, la vitesse et le bruit de la courbe de perte ?
3. Testez `sequence_len = 5` puis `sequence_len = 60`. La fenêtre optimale
   dépend-elle des fréquences présentes dans `ladybug_move` ? Que faudrait-il
   comme fenêtre pour une vraie série atmosphérique (périodes de quelques
   heures à quelques années) ?
4. Ajoutez du bruit aux positions : `dataset += np.random.normal(0, sigma, dataset.shape)` avec sigma = 0.01, 0.05, 0.1. À partir de quel niveau la prédiction casse-t-elle ?  Y-a-t il une parallèle avec la croissance des erreurs de mesure en météorologie ?

## Étape 7 — Prédiction à 1 pas

In [ ]:
s = random.randint(0, len(x_test) - sequence_len - 1)
sequence      = x_test[s:s+sequence_len]
sequence_true = x_test[s:s+sequence_len+1]

pred = model.predict(np.array([sequence]), verbose=0)[0]

plt.figure(figsize=(7, 7))
plt.plot(sequence_true[:, 0], sequence_true[:, 1], 'o-', label='vraie trajectoire')
plt.plot(pred[0], pred[1], 'o', markersize=10, label='point prédit')
plt.legend()
plt.axis('equal')
plt.show()

print('vrai :', np.round(sequence_true[-1], 3), '   prédit :', np.round(pred, 3),
      '   erreur :', np.round(np.abs(pred - sequence_true[-1]), 3))

## Étape 8 — Prédiction à n pas : le cœur de l'exercice

Pour prédire plusieurs pas d'avance, il n'existe pas de solution magique :
on **réinjecte** chaque prédiction dans la fenêtre d'entrée, qui glisse d'un cran, et on prédit le point suivant. Le réseau finit donc par se nourrir de ses propres erreurs.

In [ ]:
def predict_n_steps(model, sequence, n_steps):
    """Renvoie un tableau (n_steps, 2) des positions prédites, une par pas."""
    sequence = sequence.copy()
    predictions = []
    for _ in range(n_steps):
        pred = model.predict(np.array([sequence]), verbose=0)[0]
        predictions.append(pred)
        sequence = np.vstack([sequence[1:], pred])   # la fenêtre glisse
    return np.array(predictions)



In [ ]:
s = random.randint(0, len(x_test) - sequence_len - predict_len)
sequence      = x_test[s:s+sequence_len]
sequence_true = x_test[s:s+sequence_len+predict_len]

preds = predict_n_steps(model, sequence, predict_len)

plt.figure(figsize=(7, 7))
plt.plot(sequence_true[:, 0], sequence_true[:, 1], 'o-', label='vraie trajectoire')
plt.plot(np.vstack([sequence[-1:], preds])[:, 0],
         np.vstack([sequence[-1:], preds])[:, 1], 'o--', label='prédiction')
plt.legend()
plt.axis('equal')
plt.show()

errs = np.abs(preds - sequence_true[-predict_len:])
print('erreur par pas (x, y) :\n', np.round(errs, 3))

### EXPLORATION 4 — Horizon de prévisibilité
1. Tracez l'erreur euclidienne en fonction du pas (1 à 30). Est-elle constante, croissante, explosive ? Décrivez la forme de la courbe.
2. Tracez la prédiction à 50 pas : à partir de quand la coccinelle "prédite"
    s'arrête-t-elle dans une position quasi fixe ? Expliquez pourquoi un modèle entraîné à prédire le pas suivant converge vers la moyenne des trajectoires quand on l'itère loin.
3. Essayez 3 valeurs de `ladybug_init(s=...)` (trajectoires différentes) :
    l'horizon de prévisibilité varie-t-il ?
4. Synthèse (question de cours) : ce phénomène d'accumulation d'erreurs est
    l'analogue discret de la limite de prévisibilité de l'atmosphère (quelques    jours pour la météo synoptique, environ deux semaines au maximum pour les modèles actuels — ordre de grandeur, à nuancer selon le phénomène). Quelles différences essentielles entre notre système quasi périodique    déterministe et l'atmosphère (chaos, multi-échelles, forçage) ?



## Mini-projet
Choisissez une piste et rédigez une demi-page de conclusions chiffrées :
- Fenêtre adaptée : cherchez le `sequence_len` qui minimise l'erreur à 5 pas.
   Corrélé aux fréquences de la trajectoire ?
- Modèle minimal : quel est le plus petit réseau qui prédit encore correctement  à 1 pas ? À 5 pas ?
- Entraîner directement à 5 pas : modifiez le générateur (cible = 5 points plus loin,  via `stride` ou en décalant les cibles) et comparez avec l'approche itérative. Est-ce mieux à 5 pas ? Pourquoi la réponse n'est pas évidente ?